# Notebook 01 — whuGAIT Signal Disentanglement

The standard pipeline trains both the CNN encoder and the authentication LSTM on the same set of subjects, which makes it impossible to tell whether the membership signal in NB04 comes from CNN memorisation, LSTM memorisation, or both. This notebook sets up an experiment to separate the two contributions.

whuGAIT subjects 1–60 are assigned to three non-overlapping groups:

| Group | Subjects | CNN trains? | Auth trains? | Role |
|-------|----------|-------------|--------------|------|
| A (cnn\_ids) | 21–40 | yes | no | isolates CNN memorisation signal |
| B (auth\_ids) | 41–60 | no | yes | isolates authenticator memorisation signal |
| C (held\_out\_ids) | 1–20 | no | no | baseline (expected near random) |

D5 is not used here. D5 pairs were pre-built from all 98 whuGAIT members mixed together, so filtering them to 20-subject subgroups would leave sparse, unbalanced pairs. More importantly, D5 has a known pair-correlation asymmetry that confounds the delta signal. All pairs are built from Dataset #1 directly, which gives clean, balanced pairs for each group without those structural issues.

The `auth_pairs.npz` train split contains Group B pairs (used by NB03 for auth training). The test split contains Group A and Group C pairs interleaved; subject IDs stored in `subj1_te`/`subj2_te` let NB04 separate CNN-only from held-out deltas.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import logging
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from src.data.dataset      import load_dataset, filter_subjects
from src.data.pair_builder import build_auth_pairs

DATASET = 'whuGAIT_signal'  # <<< RUNNER INJECTS THIS

DATA_ROOT    = Path('../data')
LOG_DIR      = Path('../logs')      / DATASET
ARTIFACT_DIR = Path('../artifacts') / DATASET
RESULT_DIR   = Path('../results')   / DATASET
for _d in [LOG_DIR, ARTIFACT_DIR, RESULT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

D1_ROOT = DATA_ROOT / 'Dataset #1'

# ── Group definitions ──
CNN_IDS      = list(range(21, 41))   # 20 subjects: CNN trains on these
AUTH_IDS     = list(range(41, 61))   # 20 subjects: auth trains on these
HELD_OUT_IDS = list(range(1,  21))   # 20 subjects: held-out from both

# ── Pair counts ──
N_AUTH_TRAIN = 2_000   # pairs per class for auth training (Group B)
N_EVAL       =   500   # pairs per class per group for MIA evaluation

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('nb01_signal')
log.info('=== NB01 — whuGAIT Signal Disentanglement ===')
log.info(f'CNN group:      {len(CNN_IDS)} subjects  {CNN_IDS[0]}–{CNN_IDS[-1]}')
log.info(f'Auth group:     {len(AUTH_IDS)} subjects  {AUTH_IDS[0]}–{AUTH_IDS[-1]}')
log.info(f'Held-out group: {len(HELD_OUT_IDS)} subjects  {HELD_OUT_IDS[0]}–{HELD_OUT_IDS[-1]}')

## 1. Load Dataset #1

In [ ]:
X_d1_tr, y_d1_tr = load_dataset(str(D1_ROOT), 'train')
X_d1_te, y_d1_te = load_dataset(str(D1_ROOT), 'test')
X_d1 = np.concatenate([X_d1_tr, X_d1_te], axis=0)
y_d1 = np.concatenate([y_d1_tr, y_d1_te], axis=0)

all_ids = sorted(set(CNN_IDS + AUTH_IDS + HELD_OUT_IDS))
assert set(all_ids).issubset(set(np.unique(y_d1).tolist())), 'Some subject IDs missing from D1!'

X_cnn,   y_cnn   = filter_subjects(X_d1, y_d1, CNN_IDS)
X_auth,  y_auth  = filter_subjects(X_d1, y_d1, AUTH_IDS)
X_held,  y_held  = filter_subjects(X_d1, y_d1, HELD_OUT_IDS)

log.info(f'D1 total windows: {len(y_d1)}')
log.info(f'CNN group    ({len(CNN_IDS)} subjects): {len(y_cnn)} windows')
log.info(f'Auth group   ({len(AUTH_IDS)} subjects): {len(y_auth)} windows')
log.info(f'Held-out grp ({len(HELD_OUT_IDS)} subjects): {len(y_held)} windows')
print(f'D1 loaded: {len(y_cnn)} CNN + {len(y_auth)} auth + {len(y_held)} held-out windows')

## 2. Save Subject Split

In [ ]:
split = {
    'dataset':      DATASET,
    'train_ids':    CNN_IDS,      # NB02 reads train_ids → CNN trains on these
    'auth_ids':     AUTH_IDS,     # NB03 reads auth_ids → auth trains on these
    'held_out_ids': HELD_OUT_IDS,
    'cnn_ids':      CNN_IDS,      # explicit tag for NB04 three-way analysis
    'n_train':      len(CNN_IDS),
    'n_auth':       len(AUTH_IDS),
    'n_held_out':   len(HELD_OUT_IDS),
}

assert len(set(CNN_IDS) & set(AUTH_IDS))     == 0, 'CNN / auth overlap!'
assert len(set(CNN_IDS) & set(HELD_OUT_IDS)) == 0, 'CNN / held-out overlap!'
assert len(set(AUTH_IDS) & set(HELD_OUT_IDS)) == 0, 'Auth / held-out overlap!'

with open(ARTIFACT_DIR / 'subject_split.json', 'w') as f:
    json.dump(split, f, indent=2)

log.info('Subject split saved.')
print(f'Split saved: CNN={len(CNN_IDS)}  auth={len(AUTH_IDS)}  held-out={len(HELD_OUT_IDS)}  (no overlap ✓)')

## 3. Build Authentication Pairs

**Training pairs** come from Group B (auth subjects 41–60). These are the only subjects the authenticator LSTM trains on. Because the CNN encoder was trained on Group A (21–40) and not Group B, any memorisation signal on Group B pairs can be attributed to the authenticator rather than the CNN.

**Evaluation pairs** are built for all three groups so NB04 can score each group with the trained auth model and compute per-group deltas. The test split in `auth_pairs.npz` combines Group A and Group C pairs; subject IDs in `subj1_te`/`subj2_te` let NB04 separate them.

In [ ]:
# ── Auth training pairs (Group B only) ──
X1_tr, X2_tr, y_tr, subj1_tr, subj2_tr = build_auth_pairs(
    X_auth, y_auth, n_pairs_per_class=N_AUTH_TRAIN, seed=42)

log.info(f'Auth training pairs: {len(y_tr)}  same={int((y_tr==0).sum())}  diff={int((y_tr==1).sum())}')
print(f'Auth training pairs: {len(y_tr)} total')

In [ ]:
# ── Eval pairs for Group A (CNN-only) ──
X1_cnn_e, X2_cnn_e, y_cnn_e, s1_cnn_e, s2_cnn_e = build_auth_pairs(
    X_cnn, y_cnn, n_pairs_per_class=N_EVAL, seed=43)

# ── Eval pairs for Group C (held-out) ──
X1_held_e, X2_held_e, y_held_e, s1_held_e, s2_held_e = build_auth_pairs(
    X_held, y_held, n_pairs_per_class=N_EVAL, seed=44)

# ── Also build auth-group eval pairs (for in-sample delta) ──
X1_auth_e, X2_auth_e, y_auth_e, s1_auth_e, s2_auth_e = build_auth_pairs(
    X_auth, y_auth, n_pairs_per_class=N_EVAL, seed=45)

# Test split = CNN eval + held-out eval (the groups NB04 needs to score against)
X1_te    = np.concatenate([X1_cnn_e,  X1_held_e],  axis=0)
X2_te    = np.concatenate([X2_cnn_e,  X2_held_e],  axis=0)
y_te     = np.concatenate([y_cnn_e,   y_held_e],   axis=0)
subj1_te = np.concatenate([s1_cnn_e,  s1_held_e],  axis=0)
subj2_te = np.concatenate([s2_cnn_e,  s2_held_e],  axis=0)

log.info(f'Eval pairs — CNN group: {len(y_cnn_e)}  held-out: {len(y_held_e)}  test total: {len(y_te)}')
log.info(f'Eval pairs — auth group (in-sample): {len(y_auth_e)}')
print(f'Eval — CNN: {len(y_cnn_e)}  held-out: {len(y_held_e)}  auth in-sample: {len(y_auth_e)}')

In [ ]:
np.savez_compressed(
    ARTIFACT_DIR / 'auth_pairs.npz',
    # Training split (Group B — auth members)
    X1_tr=X1_tr, X2_tr=X2_tr, y_tr=y_tr,
    subj1_tr=subj1_tr.astype(np.int32), subj2_tr=subj2_tr.astype(np.int32),
    # Test split (Group A + Group C — for NB04 multi-group scoring)
    X1_te=X1_te, X2_te=X2_te, y_te=y_te,
    subj1_te=subj1_te.astype(np.int32), subj2_te=subj2_te.astype(np.int32),
    # Auth-group eval (Group B scored in-sample — train delta reference)
    X1_auth_eval=X1_auth_e, X2_auth_eval=X2_auth_e, y_auth_eval=y_auth_e,
    subj1_auth_eval=s1_auth_e.astype(np.int32), subj2_auth_eval=s2_auth_e.astype(np.int32),
)
log.info(f'Saved: {ARTIFACT_DIR}/auth_pairs.npz')
print('auth_pairs.npz saved ✓')

## 4. Data Exploration

In [ ]:
counts = {sid: int((y_d1 == sid).sum()) for sid in all_ids}

def color(sid):
    if sid in CNN_IDS:      return '#3498db'
    if sid in AUTH_IDS:     return '#2ecc71'
    return '#e74c3c'

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(range(len(all_ids)), [counts[s] for s in all_ids],
       color=[color(s) for s in all_ids], width=0.8)
ax.set_xticks(range(len(all_ids)))
ax.set_xticklabels(all_ids, fontsize=7, rotation=90)
ax.set_xlabel('Subject ID')
ax.set_ylabel('Window count')
ax.set_title('D1 window counts by group (subjects 1–60)')
legend_patches = [
    mpatches.Patch(color='#3498db', label=f'CNN group (21–40, n={len(CNN_IDS)})'),
    mpatches.Patch(color='#2ecc71', label=f'Auth group (41–60, n={len(AUTH_IDS)})'),
    mpatches.Patch(color='#e74c3c', label=f'Held-out (1–20, n={len(HELD_OUT_IDS)})'),
]
ax.legend(handles=legend_patches)
plt.tight_layout()
plt.savefig(RESULT_DIR / '01_signal_window_counts.png', dpi=150)
plt.show()
log.info('Figure saved: 01_signal_window_counts.png')

In [ ]:
summary = f"""
=== NB01 SIGNAL SUMMARY ===

Groups (all from Dataset #1, D5 not used):
  CNN group    (A): subjects {CNN_IDS[0]}–{CNN_IDS[-1]}   n={len(CNN_IDS)}  windows={len(y_cnn)}
  Auth group   (B): subjects {AUTH_IDS[0]}–{AUTH_IDS[-1]}   n={len(AUTH_IDS)}  windows={len(y_auth)}
  Held-out     (C): subjects {HELD_OUT_IDS[0]}–{HELD_OUT_IDS[-1]}   n={len(HELD_OUT_IDS)}  windows={len(y_held)}

Pairs saved to auth_pairs.npz:
  Train (Group B auth pairs): {len(y_tr)}
  Test  (Group A + C eval):   {len(y_te)}
  Auth eval (Group B in-sample): {len(y_auth_e)}

subject_split.json:
  train_ids    = cnn_ids   → NB02 trains CNN on these
  auth_ids     = auth_ids  → NB03 trains auth on these
  held_out_ids             → NB04 baseline
  cnn_ids                  → NB04 CNN-signal group
"""
print(summary)
log.info(summary)